# 01 — Verify preprocessing

Eyeball check on the output of `scripts/preprocess.py`. Loads 3 preprocessed cases and, for each one, plots:

1. **Mid-slices** — sagittal / coronal / axial, all 4 modalities, with the label mask overlaid in colour.
2. **Intensity histograms** — per modality, before and after normalization.

The "before" histogram needs the **raw NIfTI** volumes, which the preprocessed `.npy` files no longer contain. So this notebook reads *both* the raw BraTS tree and the preprocessed output directory.

## What you are looking for

| Check | Broken looks like |
|---|---|
| Label follows the anatomy | Mask floating in background, or offset from the bright lesion |
| Channel order is t1, t1ce, t2, flair | T1CE not brighter than T1 in the enhancing rim; FLAIR not showing bright edema |
| Normalization worked | "After" histograms not centred on 0, or std far from 1 |
| Crop kept the whole brain | Brain touching or clipped at a panel edge |
| Labels are contiguous | Any value other than 0/1/2/3 in the QC table |

## Setup

`PREP_DIR` is read from the Hydra config so it stays in sync with whatever `scripts/preprocess.py` wrote. `RAW_ROOT` has no default — the config marks it mandatory (`???`) precisely so no machine-specific path is baked in. Set it below or export `BRATS_RAW`.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from hydra import compose, initialize_config_dir

from neurovision.data.brats import scan_brats_root
from neurovision.data.preprocessing import load_case_arrays
from neurovision.visualization.qc import (
    plot_case_slices,
    plot_intensity_histograms,
)

# ---------------------------------------------------------------------------
# EDIT ME: path to the raw BraTS root (the directory of case folders).
# Falls back to the BRATS_RAW environment variable if set.
# ---------------------------------------------------------------------------
RAW_ROOT = Path(os.environ.get("BRATS_RAW", "/path/to/brats2021"))

N_CASES = 3

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_DIR = str(REPO_ROOT / "configs")

with initialize_config_dir(version_base="1.3", config_dir=CONFIG_DIR):
    cfg = compose(config_name="config", overrides=[f"data.root_dir={RAW_ROOT}"])

# Config supplies the default; NEUROVISION_PREP_DIR overrides it when the
# preprocessed output lives somewhere else.
PREP_DIR = Path(os.environ.get("NEUROVISION_PREP_DIR", REPO_ROOT / cfg.data.preprocessing.out_dir))

print(f"raw root:      {RAW_ROOT}   exists={RAW_ROOT.is_dir()}")
print(f"preprocessed:  {PREP_DIR}   exists={PREP_DIR.is_dir()}")

# Fail with instructions rather than a bare FileNotFoundError three cells later.
if not RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw BraTS root not found: {RAW_ROOT}\n"
        "Set RAW_ROOT above, or export BRATS_RAW=/path/to/brats2021"
    )
if not PREP_DIR.is_dir():
    raise FileNotFoundError(
        f"Preprocessed directory not found: {PREP_DIR}\n"
        "Run scripts/preprocess.py first, e.g.\n"
        f"    python scripts/preprocess.py data.root_dir={RAW_ROOT} "
        "data.preprocessing.limit=3\n"
        "Or point NEUROVISION_PREP_DIR at an existing output directory."
    )

In [ ]:
# Match preprocessed output back to raw cases by case_id, so the "before" and
# "after" panels are guaranteed to describe the same case.
raw_cases = {c.case_id: c for c in scan_brats_root(RAW_ROOT)}

processed_ids = sorted(
    p.name for p in PREP_DIR.iterdir() if p.is_dir() and (p / "image.npy").is_file()
)
selected_ids = [cid for cid in processed_ids if cid in raw_cases][:N_CASES]

if not selected_ids:
    raise RuntimeError(
        f"No case appears in both {PREP_DIR} and {RAW_ROOT}. "
        "Run scripts/preprocess.py first, and check RAW_ROOT points at the same dataset."
    )

print(f"{len(processed_ids)} preprocessed case(s) available; inspecting {len(selected_ids)}:")
for cid in selected_ids:
    print(f"  {cid}")

In [ ]:
# Load each selected case: preprocessed arrays + meta, and the raw volumes for
# the "before" histograms.
cases = {}
for cid in selected_ids:
    case_dir = PREP_DIR / cid
    image = np.load(case_dir / "image.npy").astype(np.float32)  # float16 on disk
    label_path = case_dir / "label.npy"
    label = np.load(label_path) if label_path.is_file() else None
    meta = json.loads((case_dir / "meta.json").read_text())
    raw_image, _raw_label, raw_meta = load_case_arrays(raw_cases[cid])
    cases[cid] = {
        "image": image,
        "label": label,
        "meta": meta,
        "raw_image": raw_image,
        "raw_meta": raw_meta,
    }

print(f"loaded {len(cases)} case(s)")

## QC table

Numbers first — these catch things the eye will miss. `bbox_ok` verifies that the stored crop box reproduces the stored cropped shape; if that is ever `False`, predictions cannot be un-cropped back to the original geometry and the data is unusable for a BraTS submission.

In [ ]:
rows = []
for cid, d in cases.items():
    meta, image, label = d["meta"], d["image"], d["label"]
    nonzero = image[0][image[0] != 0]
    bbox_shape = [end - start for start, end in meta["bbox"]]
    rows.append(
        {
            "case_id": cid,
            "original": tuple(meta["original_shape"]),
            "cropped": tuple(meta["cropped_shape"]),
            "bbox_ok": bbox_shape == list(meta["cropped_shape"]),
            "spacing": tuple(meta["spacing"]),
            "label_values": sorted(np.unique(label).tolist()) if label is not None else None,
            "t1_mean": round(float(nonzero.mean()), 3),
            "t1_std": round(float(nonzero.std()), 3),
        }
    )

qc = pd.DataFrame(rows)
display(qc)

assert qc["bbox_ok"].all(), "bbox does not reproduce cropped_shape — un-crop path is broken"
print("\nAffine of the first case (sanity-check the orientation assumption):")
print(np.array(cases[selected_ids[0]]["meta"]["affine"]))

## Mid-slices with label overlay

Rows are modalities (T1, T1CE, T2, FLAIR), columns are sagittal / coronal / axial. Overlay: <span style="color:#56B4E9">**NCR/NET**</span>, <span style="color:#009E73">**ED**</span>, <span style="color:#D55E00">**ET**</span>.

The mid-slice may miss the tumour entirely in some cases — that is normal and not a bug. Check the histograms and the QC table too, not just these panels.

In [ ]:
for cid, d in cases.items():
    fig = plot_case_slices(d["image"], d["label"], case_id=cid)
    display(fig)

## Intensity histograms — before and after normalization

Left column is the raw volume, right column is after per-modality z-scoring over nonzero voxels.

**Only nonzero (brain) voxels are plotted.** MRI background is exactly 0 and is most of the volume; including it gives one huge spike at zero and flattens everything else out of view.

The four raw modalities should have visibly *different* intensity ranges — that is exactly why each is normalized independently. Every "after" panel should be centred on 0 with std ≈ 1.

In [ ]:
for cid, d in cases.items():
    fig = plot_intensity_histograms(d["raw_image"], d["image"], case_id=cid)
    display(fig)

## Verdict

Work through this list before trusting the preprocessed dataset:

- [ ] `bbox_ok` is `True` for every case (asserted above)
- [ ] `label_values` is a subset of `[0, 1, 2, 3]` — no raw label `4` survived the remap
- [ ] `t1_mean` ≈ 0.00 and `t1_std` ≈ 1.00
- [ ] Label overlay sits on the lesion, not in the background or off to one side
- [ ] T1CE shows brighter enhancement than T1 (confirms channels are not swapped)
- [ ] FLAIR shows bright peritumoral edema roughly where the <span style="color:#009E73">green ED</span> mask is
- [ ] Brain is not clipped at any panel edge
- [ ] Every "after" histogram is centred on 0

If any of these fail, stop and fix preprocessing — everything downstream inherits the problem.